#  Football Match Outcome Prediction — Data Cleaning

This notebook prepares the raw match data for modeling.  

DATA CLEANING PROCESS:
1. Fix multi-level column headers from raw scraping
2. Standardize column names to lowercase
3. Remove columns with missing values
4. Merge all statistics into single DataFrame
5. Convert data types for analysis

In [18]:
import pandas as pd

# Reading the raw CSV files

fixtures = pd.read_csv('fixtures_raw.csv')
shooting = pd.read_csv('shooting_raw.csv')
passing = pd.read_csv('passing_raw.csv')
possession = pd.read_csv('possession_raw.csv')
gsc = pd.read_csv('gsc_raw.csv')

In [19]:
pd.set_option('display.max_columns', None)
shooting.head()

,league,season,team,game,date,round,day,venue,result,GF,GA,opponent,Standard,Standard.1,Standard.2,Standard.3,Standard.4,Standard.5,Standard.6,Standard.7,Standard.8,Standard.9,Expected,Expected.1,Expected.2,Expected.3,Expected.4,time,match_report
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Gls,Sh,SoT,SoT%,G/Sh,G/SoT,Dist,FK,PK,PKatt,xG,npxG,npxG/Sh,G-xG,np:G-xG,NaN,NaN
1,ESP-La Liga,2122.0,Alavés,2021-08-14 Alavés-Real Madrid,2021-08-14,Matchweek 1,Sat,Home,L,1.0,4.0,Real Madrid,1,10,3,30.0,0.0,0.0,19.3,1,1,1,1.7,0.9,0.09,-0.7,-0.9,22:00:00,/en/matches/9613fb68/Alaves-Real-Madrid-August...
2,ESP-La Liga,2122.0,Alavés,2021-08-21 Alavés-Mallorca,2021-08-21,Matchweek 2,Sat,Home,L,0.0,1.0,Mallorca,0,9,4,44.4,0.0,0.0,14.8,0,0,0,0.5,0.5,0.06,-0.5,-0.5,17:00:00,/en/matches/b5a317ed/Alaves-Mallorca-August-21...
3,ESP-La Liga,2122.0,Alavés,2021-08-27 Valencia-Alavés,2021-08-27,Matchweek 3,Fri,Away,L,0.0,3.0,Valencia,0,9,2,22.2,0.0,0.0,19.2,1,0,0,0.4,0.4,0.05,-0.4,-0.4,22:15:00,/en/matches/3334afc1/Valencia-Alaves-August-27...
4,ESP-La Liga,2122.0,Alavés,2021-09-18 Alavés-Osasuna,2021-09-18,Matchweek 5,Sat,Home,L,0.0,2.0,Osasuna,0,8,3,37.5,0.0,0.0,22.6,0,0,0,0.5,0.5,0.06,-0.5,-0.5,21:00:00,/en/matches/623e3c01/Alaves-Osasuna-September-...


In [20]:
# Clean shooting dataframe column names:
# - Columns 12 to -2 had metric names in first row (Gls, Sh, etc.)
# - Combine original column names with these metrics (e.g., 'Standard' + 'Gls' = 'Standard_Gls')
# - Remove first row after extracting the metric names

cols_to_rename = shooting.columns[12:-2]

rename_dict = {col: f"{col}_{shooting[col].iloc[0]}" for col in cols_to_rename}

shooting = shooting.rename(columns=rename_dict)

shooting = shooting.iloc[1:].reset_index(drop=True)

shooting.head()

,league,season,team,game,date,round,day,venue,result,GF,GA,opponent,Standard_Gls,Standard.1_Sh,Standard.2_SoT,Standard.3_SoT%,Standard.4_G/Sh,Standard.5_G/SoT,Standard.6_Dist,Standard.7_FK,Standard.8_PK,Standard.9_PKatt,Expected_xG,Expected.1_npxG,Expected.2_npxG/Sh,Expected.3_G-xG,Expected.4_np:G-xG,time,match_report
0,ESP-La Liga,2122.0,Alavés,2021-08-14 Alavés-Real Madrid,2021-08-14,Matchweek 1,Sat,Home,L,1.0,4.0,Real Madrid,1,10,3,30.0,0.0,0.0,19.3,1,1,1,1.7,0.9,0.09,-0.7,-0.9,22:00:00,/en/matches/9613fb68/Alaves-Real-Madrid-August...
1,ESP-La Liga,2122.0,Alavés,2021-08-21 Alavés-Mallorca,2021-08-21,Matchweek 2,Sat,Home,L,0.0,1.0,Mallorca,0,9,4,44.4,0.0,0.0,14.8,0,0,0,0.5,0.5,0.06,-0.5,-0.5,17:00:00,/en/matches/b5a317ed/Alaves-Mallorca-August-21...
2,ESP-La Liga,2122.0,Alavés,2021-08-27 Valencia-Alavés,2021-08-27,Matchweek 3,Fri,Away,L,0.0,3.0,Valencia,0,9,2,22.2,0.0,0.0,19.2,1,0,0,0.4,0.4,0.05,-0.4,-0.4,22:15:00,/en/matches/3334afc1/Valencia-Alaves-August-27...
3,ESP-La Liga,2122.0,Alavés,2021-09-18 Alavés-Osasuna,2021-09-18,Matchweek 5,Sat,Home,L,0.0,2.0,Osasuna,0,8,3,37.5,0.0,0.0,22.6,0,0,0,0.5,0.5,0.06,-0.5,-0.5,21:00:00,/en/matches/623e3c01/Alaves-Osasuna-September-...
4,ESP-La Liga,2122.0,Alavés,2021-09-22 Espanyol-Alavés,2021-09-22,Matchweek 6,Wed,Away,L,0.0,1.0,Espanyol,0,7,2,28.6,0.0,0.0,14.9,0,0,0,0.5,0.5,0.07,-0.5,-0.5,19:30:00,/en/matches/a62f546a/Espanyol-Alaves-September...


In [21]:
# Rename passing columns (index 12 to -10) by appending first row values, then drop first row

cols_to_rename_2 = passing.columns[12:-10]

rename_dict_2 = {col: f"{col}_{passing[col].iloc[0]}" for col in cols_to_rename_2}

passing = passing.rename(columns=rename_dict_2)

passing = passing.iloc[1:].reset_index(drop=True)

passing.head()

,league,season,team,game,date,round,day,venue,result,GF,GA,opponent,Total_Cmp,Total.1_Att,Total.2_Cmp%,Total.3_TotDist,Total.4_PrgDist,Short_Cmp,Short.1_Att,Short.2_Cmp%,Medium_Cmp,Medium.1_Att,Medium.2_Cmp%,Long_Cmp,Long.1_Att,Long.2_Cmp%,Ast,xAG,xA,KP,1/3,PPA,CrsPA,PrgP,time,match_report
0,ESP-La Liga,2122.0,Alavés,2021-08-14 Alavés-Real Madrid,2021-08-14,Matchweek 1,Sat,Home,L,1.0,4.0,Real Madrid,360,439,82.0,6508,2692,176,198,88.9,119,140,85.0,52,69,75.4,0.0,0.7,0.3,7.0,19.0,6.0,5.0,20.0,22:00:00,/en/matches/9613fb68/Alaves-Real-Madrid-August...
1,ESP-La Liga,2122.0,Alavés,2021-08-21 Alavés-Mallorca,2021-08-21,Matchweek 2,Sat,Home,L,0.0,1.0,Mallorca,253,349,72.5,5076,1889,103,120,85.8,102,123,82.9,40,87,46.0,0.0,0.3,0.5,6.0,16.0,4.0,1.0,21.0,17:00:00,/en/matches/b5a317ed/Alaves-Mallorca-August-21...
2,ESP-La Liga,2122.0,Alavés,2021-08-27 Valencia-Alavés,2021-08-27,Matchweek 3,Fri,Away,L,0.0,3.0,Valencia,309,420,73.6,6235,2481,124,146,84.9,123,163,75.5,51,84,60.7,0.0,0.4,0.3,8.0,22.0,4.0,1.0,24.0,22:15:00,/en/matches/3334afc1/Valencia-Alaves-August-27...
3,ESP-La Liga,2122.0,Alavés,2021-09-18 Alavés-Osasuna,2021-09-18,Matchweek 5,Sat,Home,L,0.0,2.0,Osasuna,354,508,69.7,6527,2789,166,206,80.6,151,183,82.5,36,93,38.7,0.0,0.1,0.8,4.0,25.0,2.0,0.0,28.0,21:00:00,/en/matches/623e3c01/Alaves-Osasuna-September-...
4,ESP-La Liga,2122.0,Alavés,2021-09-22 Espanyol-Alavés,2021-09-22,Matchweek 6,Wed,Away,L,0.0,1.0,Espanyol,307,420,73.1,5995,2505,140,163,85.9,110,144,76.4,50,86,58.1,0.0,0.4,0.6,5.0,38.0,11.0,5.0,39.0,19:30:00,/en/matches/a62f546a/Espanyol-Alaves-September...


In [22]:
# Rename possession columns (index 13 to -2) by appending first row values, then drop first row

cols_to_rename_3 = possession.columns[13:-2]

rename_dict_3 = {col: f"{col}_{possession[col].iloc[0]}" for col in cols_to_rename_3}

possession = possession.rename(columns=rename_dict_3)

possession = possession.iloc[1:].reset_index(drop=True)

possession.head()

,league,season,team,game,date,round,day,venue,result,GF,GA,opponent,Poss,Touches_Touches,Touches.1_Def Pen,Touches.2_Def 3rd,Touches.3_Mid 3rd,Touches.4_Att 3rd,Touches.5_Att Pen,Touches.6_Live,Take-Ons_Att,Take-Ons.1_Succ,Take-Ons.2_Succ%,Take-Ons.3_Tkld,Take-Ons.4_Tkld%,Carries_Carries,Carries.1_TotDist,Carries.2_PrgDist,Carries.3_PrgC,Carries.4_1/3,Carries.5_CPA,Carries.6_Mis,Carries.7_Dis,Receiving_Rec,Receiving.1_PrgR,time,match_report
0,ESP-La Liga,2122.0,Alavés,2021-08-14 Alavés-Real Madrid,2021-08-14,Matchweek 1,Sat,Home,L,1.0,4.0,Real Madrid,42.0,526,65,192,217,121,11,525,7,4,57.1,3,42.9,292,1562,749,14,16,3,20,7,360,20,22:00:00,/en/matches/9613fb68/Alaves-Real-Madrid-August...
1,ESP-La Liga,2122.0,Alavés,2021-08-21 Alavés-Mallorca,2021-08-21,Matchweek 2,Sat,Home,L,0.0,1.0,Mallorca,46.0,461,63,174,206,89,16,461,10,9,90.0,1,10.0,232,1156,521,9,10,3,23,7,251,20,17:00:00,/en/matches/b5a317ed/Alaves-Mallorca-August-21...
2,ESP-La Liga,2122.0,Alavés,2021-08-27 Valencia-Alavés,2021-08-27,Matchweek 3,Fri,Away,L,0.0,3.0,Valencia,51.0,506,57,173,221,116,16,506,20,13,65.0,7,35.0,242,1375,587,12,6,2,20,7,308,24,22:15:00,/en/matches/3334afc1/Valencia-Alaves-August-27...
3,ESP-La Liga,2122.0,Alavés,2021-09-18 Alavés-Osasuna,2021-09-18,Matchweek 5,Sat,Home,L,0.0,2.0,Osasuna,55.0,626,58,205,280,145,18,626,21,8,38.1,13,61.9,322,1510,703,17,13,5,21,13,348,28,21:00:00,/en/matches/623e3c01/Alaves-Osasuna-September-...
4,ESP-La Liga,2122.0,Alavés,2021-09-22 Espanyol-Alavés,2021-09-22,Matchweek 6,Wed,Away,L,0.0,1.0,Espanyol,47.0,506,40,124,253,132,26,506,8,3,37.5,5,62.5,230,1320,671,16,7,5,17,6,301,36,19:30:00,/en/matches/a62f546a/Espanyol-Alaves-September...


In [23]:
# Rename gsc columns (index 12 to -2) by appending first row values, then drop first row

cols_to_rename_4 = gsc.columns[12:-2]

rename_dict_4 = {col: f"{col}_{gsc[col].iloc[0]}" for col in cols_to_rename_4}

gsc = gsc.rename(columns=rename_dict_4)

gsc = gsc.iloc[1:].reset_index(drop=True)

gsc.head()

,league,season,team,game,date,round,day,venue,result,GF,GA,opponent,SCA Types_SCA,SCA Types.1_PassLive,SCA Types.2_PassDead,SCA Types.3_TO,SCA Types.4_Sh,SCA Types.5_Fld,SCA Types.6_Def,GCA Types_GCA,GCA Types.1_PassLive,GCA Types.2_PassDead,GCA Types.3_TO,GCA Types.4_Sh,GCA Types.5_Fld,GCA Types.6_Def,time,match_report
0,ESP-La Liga,2122.0,Alavés,2021-08-14 Alavés-Real Madrid,2021-08-14,Matchweek 1,Sat,Home,L,1.0,4.0,Real Madrid,18,15,0,1,1,1,0,1,0,0,0,0,1,0,22:00:00,/en/matches/9613fb68/Alaves-Real-Madrid-August...
1,ESP-La Liga,2122.0,Alavés,2021-08-21 Alavés-Mallorca,2021-08-21,Matchweek 2,Sat,Home,L,0.0,1.0,Mallorca,18,9,3,3,0,2,1,0,0,0,0,0,0,0,17:00:00,/en/matches/b5a317ed/Alaves-Mallorca-August-21...
2,ESP-La Liga,2122.0,Alavés,2021-08-27 Valencia-Alavés,2021-08-27,Matchweek 3,Fri,Away,L,0.0,3.0,Valencia,15,6,4,1,1,3,0,0,0,0,0,0,0,0,22:15:00,/en/matches/3334afc1/Valencia-Alaves-August-27...
3,ESP-La Liga,2122.0,Alavés,2021-09-18 Alavés-Osasuna,2021-09-18,Matchweek 5,Sat,Home,L,0.0,2.0,Osasuna,8,3,2,0,1,1,1,0,0,0,0,0,0,0,21:00:00,/en/matches/623e3c01/Alaves-Osasuna-September-...
4,ESP-La Liga,2122.0,Alavés,2021-09-22 Espanyol-Alavés,2021-09-22,Matchweek 6,Wed,Away,L,0.0,1.0,Espanyol,13,9,2,0,2,0,0,0,0,0,0,0,0,0,19:30:00,/en/matches/a62f546a/Espanyol-Alaves-September...


In [24]:
# Convert all column names to lowercase for all 5 dataframes

dataframes = [fixtures, shooting, passing, possession, gsc]

for df in dataframes:
    df.columns = df.columns.str.lower()

In [25]:
# Dropping "notes" column from the fixtures Dataframe because it only contains missing values
# Dropping 5 columns from the shooting Dataframe to eliminate the missing values

fixtures = fixtures.drop(columns=['notes'])

shooting = shooting.drop(columns=[
    'standard.3_sot%', 'standard.4_g/sh', 'standard.5_g/sot', 'standard.6_dist', 'expected.2_npxg/sh'
])


In [26]:
# Export cleaned CSVs
fixtures.to_csv('fixtures_clean.csv', index=False)
shooting.to_csv('shooting_clean.csv', index=False)
passing.to_csv('passing_clean.csv', index=False)
possession.to_csv('possession_clean.csv', index=False)
gsc.to_csv('gsc_clean.csv', index=False)

In [27]:
# Merge all dataframes (fixtures, possession, shooting, gsc, passing) on common columns

from functools import reduce

# List of dataframes to merge

dfs = [fixtures, possession, shooting, gsc, passing]

# Find common columns between all dataframes for merging

common_cols = set(dfs[0].columns)
for df in dfs[1:]:
    common_cols &= set(df.columns)

print("Common columns for merging:", common_cols)

# Merge all using reduce

merged_df = reduce(lambda left, right: pd.merge(
    left, right, how='outer', on=list(common_cols), suffixes=('', '_dup')
), dfs)


Common columns for merging: {'round', 'day', 'league', 'venue', 'date', 'ga', 'match_report', 'season', 'time', 'opponent', 'game', 'gf', 'result', 'team'}


In [28]:
# Remove duplicate columns created during merge

merged = merged_df.loc[:, ~merged_df.columns.duplicated()]

In [29]:
merged.columns

Index(['league', 'season', 'team', 'game', 'date', 'time', 'round', 'day',
       'venue', 'result', 'gf', 'ga', 'opponent', 'xg', 'xga', 'poss',
       'attendance', 'captain', 'formation', 'opp formation', 'referee',
       'match_report', 'poss_dup', 'touches_touches', 'touches.1_def pen',
       'touches.2_def 3rd', 'touches.3_mid 3rd', 'touches.4_att 3rd',
       'touches.5_att pen', 'touches.6_live', 'take-ons_att',
       'take-ons.1_succ', 'take-ons.2_succ%', 'take-ons.3_tkld',
       'take-ons.4_tkld%', 'carries_carries', 'carries.1_totdist',
       'carries.2_prgdist', 'carries.3_prgc', 'carries.4_1/3', 'carries.5_cpa',
       'carries.6_mis', 'carries.7_dis', 'receiving_rec', 'receiving.1_prgr',
       'standard_gls', 'standard.1_sh', 'standard.2_sot', 'standard.7_fk',
       'standard.8_pk', 'standard.9_pkatt', 'expected_xg', 'expected.1_npxg',
       'expected.3_g-xg', 'expected.4_np:g-xg', 'sca types_sca',
       'sca types.1_passlive', 'sca types.2_passdead', 'sca types.

In [30]:
# Convert statistical columns from object type to numeric for analysis

# List of columns to convert

numeric_columns = [
    'touches_touches', 'touches.1_def pen', 'touches.2_def 3rd',
    'touches.3_mid 3rd', 'touches.4_att 3rd', 'touches.5_att pen',
    'touches.6_live', 'take-ons_att', 'take-ons.1_succ', 'take-ons.2_succ%',
    'take-ons.3_tkld', 'take-ons.4_tkld%', 'carries_carries',
    'carries.1_totdist', 'carries.2_prgdist', 'carries.3_prgc',
    'carries.4_1/3', 'carries.5_cpa', 'carries.6_mis', 'carries.7_dis',
    'receiving_rec', 'receiving.1_prgr', 'standard_gls', 'standard.1_sh',
    'standard.2_sot', 'standard.7_fk', 'standard.8_pk', 'standard.9_pkatt',
    'expected_xg', 'expected.1_npxg', 'expected.3_g-xg', 'expected.4_np:g-xg',
    'sca types_sca', 'sca types.1_passlive', 'sca types.2_passdead',
    'sca types.3_to', 'sca types.4_sh', 'sca types.5_fld', 'sca types.6_def',
    'gca types_gca', 'gca types.1_passlive', 'gca types.2_passdead',
    'gca types.3_to', 'gca types.4_sh', 'gca types.5_fld', 'gca types.6_def',
    'total_cmp', 'total.1_att', 'total.2_cmp%', 'total.3_totdist',
    'total.4_prgdist', 'short_cmp', 'short.1_att', 'short.2_cmp%',
    'medium_cmp', 'medium.1_att', 'medium.2_cmp%', 'long_cmp',
    'long.1_att', 'long.2_cmp%'
]

# Convert all columns at once

for col in numeric_columns:
    merged[col] = pd.to_numeric(merged[col], errors='coerce')

# Check for any conversion issues

print(merged[numeric_columns].info())
print(merged[numeric_columns].isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3040 entries, 0 to 3039
Data columns (total 60 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   touches_touches       3040 non-null   int64  
 1   touches.1_def pen     3040 non-null   int64  
 2   touches.2_def 3rd     3040 non-null   int64  
 3   touches.3_mid 3rd     3040 non-null   int64  
 4   touches.4_att 3rd     3040 non-null   int64  
 5   touches.5_att pen     3040 non-null   int64  
 6   touches.6_live        3040 non-null   int64  
 7   take-ons_att          3040 non-null   int64  
 8   take-ons.1_succ       3040 non-null   int64  
 9   take-ons.2_succ%      3040 non-null   float64
 10  take-ons.3_tkld       3040 non-null   int64  
 11  take-ons.4_tkld%      3040 non-null   float64
 12  carries_carries       3040 non-null   int64  
 13  carries.1_totdist     3040 non-null   int64  
 14  carries.2_prgdist     3040 non-null   int64  
 15  carries.3_prgc       

In [31]:
# Export merged dataframe as CSV
merged.to_csv('merged.csv', index=False)